In [ ]:
!pip install faiss-cpu
!pip install google-generativeai
!pip install yfinance

In [ ]:
!pip install langchain-text-splitters

In [ ]:
import pandas as pd
import numpy as np
from google import genai ### communication with Gemini API
import faiss ## vector store
import yfinance as yf ## stock data
from sentence_transformers import SentenceTransformer ## sentence embedding
from langchain_text_splitters import RecursiveCharacterTextSplitter ## text splitter

In [ ]:
ticker = "RELIANCE.NS"
stock = yf.Ticker(ticker)

In [ ]:
from pandas._libs.tslibs import period
historical_data = stock.history(period = "6mo")
print(historical_data.tail())
print("--------------------")
print(historical_data.head())

In [ ]:
company_info = stock.info
print(company_info)

In [ ]:
news = stock.news
print(news)

In [ ]:
for item in news[:5]:
  print(item['content']['title'])
  print(item['content']['summary'])

In [ ]:
### Create a Financial Knowledge base
### Combine Historical Data + Company Info + News
documents = []
Company_text = f"""
Industry : {company_info['industry']}
Sector : {company_info['sector']}
Description : {company_info['longBusinessSummary']}
"""
documents.append(Company_text)
print(documents)


In [ ]:
for item in news[:5]:
  news_text = f"""
  Title :{item['content']['title']}
  Summary :{item['content']['summary']}
  """
  documents.append(news_text)
  print(documents)


In [ ]:
#### Historical Data
historical_data_text = historical_data.tail(30).to_string()
documents.append(historical_data_text)
print(documents)


In [ ]:
print(documents[0])

In [ ]:
#### Text Splitting - Chunking
text_splitter =  RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 0)
chunks = text_splitter.create_documents(documents)
print(chunks)

In [ ]:
print(len(chunks))

In [ ]:
print(chunks[5])

In [ ]:
#### Embeddings of the Chunks -
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
text = [doc.page_content for doc in chunks]
embeddings =  embedding_model.encode(text)
print(embeddings)

In [ ]:
print(embeddings.shape)

In [ ]:
!pip install -q yfinance faiss-cpu sentence-transformers

In [ ]:
import pandas as pd
import numpy as np
from google import genai ### communication with Gemini API
import faiss ## vector store
import yfinance as yf ## stock data
from sentence_transformers import SentenceTransformer ## sentence embedding


In [ ]:
ticker = yf.Ticker("RELIANCE.NS")

In [ ]:
info = ticker.info

In [ ]:
print("Company:", info.get("longName", "Reliance Industries Limited"))
print("Sector:", info.get("sector", "Not available"))
print("Industry:", info.get("industry", "Not available"))
print("Currency:", info.get("currency", "Not available"))


In [ ]:
income_stmt = ticker.income_stmt
balance_sheet = ticker.balance_sheet
cash_flow = ticker.cashflow

print("\nIncome statement shape:", income_stmt.shape)
print("Balance sheet shape:", balance_sheet.shape)
print("Cash flow shape:", cash_flow.shape)

In [ ]:
display(income_stmt.head())


In [ ]:
documents = []

In [ ]:
profile_fields = [
    "longName", "sector", "industry", "longBusinessSummary",
    "marketCap", "trailingPE", "priceToBook",
    "returnOnEquity", "profitMargins", "totalDebt",
    "totalCash", "dividendYield"
]

profile_text = []

for field in profile_fields:
    value = info.get(field)
    if value is not None:
        profile_text.append(f"{field}: {value}")


In [ ]:
if profile_text:
    documents.append({
        "source": "Yahoo Finance - Company Profile",
        "text": "\n".join(profile_text)
    })


def statement_to_documents(statement, statement_name):
    docs = []

    if statement is None or statement.empty:
        return docs



In [ ]:
import pandas as pd

def statement_to_documents(statement, statement_name):
    docs = []

    if statement is None or statement.empty:
        print(f"No data available for {statement_name}")
        return docs

    for metric, row in statement.iterrows():
        for period, value in row.items():

            if pd.isna(value):
                continue

            period_text = pd.Timestamp(period).strftime("%Y-%m-%d")

            docs.append({
                "source": f"Yahoo Finance - {statement_name}",
                "text": (
                    f"Reliance Industries Limited. "
                    f"Statement: {statement_name}. "
                    f"Metric: {metric}. "
                    f"Reporting period: {period_text}. "
                    f"Value: {value}."
                )
            })

    return docs


# Call the function with your actual financial statements
documents = []

documents += statement_to_documents(
    income_stmt, "Income Statement"
)

documents += statement_to_documents(
    balance_sheet, "Balance Sheet"
)

documents += statement_to_documents(
    cash_flow, "Cash Flow Statement"
)

print("Total passages created:", len(documents))

# Display sample passages
for doc in documents[:5]:
    print(doc["text"])


In [ ]:
ticker = yf.Ticker("RELIANCE.NS")

income_stmt = ticker.income_stmt
balance_sheet = ticker.balance_sheet
cash_flow = ticker.cashflow

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

if not documents:
    raise ValueError(
        "No financial data was retrieved. Check the Yahoo Finance response."
    )

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

texts = [doc["text"] for doc in documents]

embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
).astype("float32")

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print("FAISS index created.")
print("Indexed passages:", index.ntotal)


In [ ]:
def search_financial_data(question, top_k=5):
    query_vector = model.encode(
        [question],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    k = min(top_k, index.ntotal)
    scores, ids = index.search(query_vector, k)

    results = []

    for score, idx in zip(scores[0], ids[0]):
        if idx >= 0:
            results.append({
                "score": round(float(score), 4),
                "source": documents[int(idx)]["source"],
                "text": documents[int(idx)]["text"]
            })

    return results


question = (
    "How have Reliance Industries' revenue and net income "
    "changed across financial years?"
)

results = search_financial_data(question)

for result in results:
    print("Source:", result["source"])
    print("Similarity:", result["score"])
    print("Retrieved data:", result["text"])
    print("-" * 60)


In [ ]:
%pip install -q -U google-genai

In [ ]:
!pip install -q -U google-genai

from google import genai
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

print("Gemini client initialized successfully!")

In [ ]:
from google import genai
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")

if not api_key:
    raise ValueError(
        "Add GEMINI_API_KEY to Google Colab Secrets first."
    )

client = genai.Client(api_key=api_key)

print("Gemini client initialized successfully.")


In [ ]:
from google import genai

def ask_reliance_rag(question, top_k=5):

    # 1. Retrieve relevant financial information from FAISS
    results = search_financial_data(question, top_k=top_k)

    if not results:
        return "No relevant financial information was found."

    # 2. Prepare the retrieved evidence
    context_parts = []

    for i, result in enumerate(results, start=1):
        context_parts.append(
            f"[Evidence {i}]\n"
            f"Source: {result['source']}\n"
            f"Similarity: {result['score']}\n"
            f"Financial information: {result['text']}"
        )

    context = "\n\n".join(context_parts)

    # 3. Create the prompt for Gemini
    prompt = f"""
You are a financial analysis assistant for Reliance Industries Limited.

Answer the question using only the retrieved financial information below.

Instructions:
- Do not invent numbers or financial facts.
- Mention reporting periods when available.
- Cite claims using [Evidence 1], [Evidence 2], etc.
- Explain when the available evidence is insufficient.
- Separate reported facts from your interpretation.
- Do not provide a buy, sell, or hold recommendation.

RETRIEVED FINANCIAL INFORMATION:
{context}

QUESTION:
{question}

Write a clear answer with headings and bullet points where useful.
"""

    # 4. Generate the response using Gemini
    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt
    )

    # 5. Return the generated answer
    if response.text:
        return response.text

    return "Gemini did not return a text response."


# 6. Test your RAG system
question = "How is Reliance Industries performing in terms of debt and cash flow?"

answer = ask_reliance_rag(question)

print("QUESTION:")
print(question)

print("\nGEMINI RAG RESPONSE:")
print(answer)


In [ ]:
%pip install -q gradio

In [ ]:
import gradio as gr

def chat_with_reliance(question):
    try:
        return reliance_financial_qa(question, top_k=5)
    except Exception as e:
        return (
            f"An error occurred: {e}\n\n"
            "Please check that your Gemini client, FAISS index, "
            "and Yahoo Finance data are initialized."
        )


demo = gr.Interface(
    fn=chat_with_reliance,
    inputs=gr.Textbox(
        lines=3,
        label="Ask about Reliance Industries Limited",
        placeholder=(
            "Example: How has revenue changed across "
            "the available reporting periods?"
        )
    ),
    outputs=gr.Textbox(
        lines=18,
        label="Financial analysis and retrieved evidence"
    ),
    title="Reliance Industries — AI Financial Research Assistant",
    description=(
        "Ask natural-language questions about Reliance Industries. "
        "Answers are generated by Gemini using information retrieved "
        "from your Yahoo Finance data indexed in FAISS."
    ),
    examples=[
        "What financial information is available about profitability?",
        "What does the available data indicate about total debt?",
        "How does operating cash flow compare with net income?",
        "What are the limitations of the available financial data?"
    ]
)

demo.launch()


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

ticker = yf.Ticker("RELIANCE.NS")

income = ticker.income_stmt
balance = ticker.balance_sheet
cashflow = ticker.cashflow
info = ticker.info


def get_metric(statement, names):
    """Find the first available financial metric."""
    if statement is None or statement.empty:
        return None

    for name in names:
        if name in statement.index:
            return statement.loc[name]

    return None


def get_period_value(series, period):
    if series is None or period not in series.index:
        return np.nan

    value = pd.to_numeric(series.loc[period], errors="coerce")

    if pd.isna(value):
        return np.nan

    return float(value)


# Income statement metrics
revenue = get_metric(income, ["Total Revenue", "Operating Revenue"])
net_income = get_metric(
    income, ["Net Income", "Net Income Common Stockholders"]
)
operating_income = get_metric(income, ["Operating Income"])
basic_eps = get_metric(income, ["Basic EPS"])
diluted_eps = get_metric(income, ["Diluted EPS"])

# Balance sheet metrics
assets = get_metric(balance, ["Total Assets"])
liabilities = get_metric(
    balance, ["Total Liabilities Net Minority Interest", "Total Liabilities"]
)
equity = get_metric(
    balance, ["Stockholders Equity", "Total Stockholder Equity",
              "Common Stock Equity"]
)
debt = get_metric(balance, ["Total Debt"])
cash = get_metric(
    balance, ["Cash And Cash Equivalents",
              "Cash Cash Equivalents And Short Term Investments"]
)
current_assets = get_metric(balance, ["Current Assets"])
current_liabilities = get_metric(balance, ["Current Liabilities"])

# Cash flow metrics
operating_cf = get_metric(
    cashflow, ["Operating Cash Flow", "Cash Flow From Continuing Operating Activities"]
)
capex = get_metric(cashflow, ["Capital Expenditure"])

# Include all available reporting periods
periods = set()

for statement in [income, balance, cashflow]:
    if statement is not None and not statement.empty:
        periods.update(statement.columns)

periods = sorted(periods)

rows = []

for period in periods:
    rev = get_period_value(revenue, period)
    ni = get_period_value(net_income, period)
    oi = get_period_value(operating_income, period)
    eps_basic = get_period_value(basic_eps, period)
    eps_diluted = get_period_value(diluted_eps, period)

    total_assets = get_period_value(assets, period)
    total_liabilities = get_period_value(liabilities, period)
    total_equity = get_period_value(equity, period)
    total_debt = get_period_value(debt, period)
    cash_value = get_period_value(cash, period)
    ca = get_period_value(current_assets, period)
    cl = get_period_value(current_liabilities, period)

    ocf = get_period_value(operating_cf, period)
    capital_expenditure = get_period_value(capex, period)

    # Free cash flow: operating cash flow minus absolute capex
    fcf = (
        ocf - abs(capital_expenditure)
        if pd.notna(ocf) and pd.notna(capital_expenditure)
        else np.nan
    )

    rows.append({
        "Period": pd.Timestamp(period),
        "Revenue": rev,
        "Net Income": ni,
        "Operating Income": oi,
        "Basic EPS": eps_basic,
        "Diluted EPS": eps_diluted,
        "Total Assets": total_assets,
        "Total Liabilities": total_liabilities,
        "Shareholders Equity": total_equity,
        "Total Debt": total_debt,
        "Cash & Equivalents": cash_value,
        "Net Debt": (
            total_debt - cash_value
            if pd.notna(total_debt) and pd.notna(cash_value)
            else np.nan
        ),
        "Operating Cash Flow": ocf,
        "Capital Expenditure": capital_expenditure,
        "Free Cash Flow": fcf,
        "Net Profit Margin (%)": (
            ni / rev * 100
            if pd.notna(ni) and pd.notna(rev) and rev != 0
            else np.nan
        ),
        "Operating Margin (%)": (
            oi / rev * 100
            if pd.notna(oi) and pd.notna(rev) and rev != 0
            else np.nan
        ),
        "Debt-to-Equity (%)": (
            total_debt / total_equity * 100
            if pd.notna(total_debt) and pd.notna(total_equity)
            and total_equity != 0
            else np.nan
        ),
        "Current Ratio": (
            ca / cl
            if pd.notna(ca) and pd.notna(cl) and cl != 0
            else np.nan
        ),
        "ROE (%)": (
            ni / total_equity * 100
            if pd.notna(ni) and pd.notna(total_equity)
            and total_equity != 0
            else np.nan
        )
    })

fundamentals = pd.DataFrame(rows).sort_values("Period").reset_index(drop=True)

# Year-over-year growth: interpret cautiously if periods are not comparable
fundamentals["Revenue Growth (%)"] = (
    fundamentals["Revenue"].pct_change(fill_method=None) * 100
)
fundamentals["Net Income Growth (%)"] = (
    fundamentals["Net Income"].pct_change(fill_method=None) * 100
)
fundamentals["EPS Growth (%)"] = (
    fundamentals["Diluted EPS"].pct_change(fill_method=None) * 100
)

fundamentals["Period"] = fundamentals["Period"].dt.strftime("%Y-%m-%d")

print("Reliance Industries Limited — Fundamental Metrics")
print("Financial statement values are generally reported in INR.")
display(fundamentals.round(2))


In [ ]:
valuation_fields = {
    "Current Price": "currentPrice",
    "Market Capitalization": "marketCap",
    "Trailing P/E": "trailingPE",
    "Forward P/E": "forwardPE",
    "Price-to-Book": "priceToBook",
    "Return on Equity": "returnOnEquity",
    "Profit Margin": "profitMargins",
    "Dividend Yield": "dividendYield",
    "Enterprise Value": "enterpriseValue",
    "Enterprise Value / EBITDA": "enterpriseToEbitda"
}

valuation = {}

for label, field in valuation_fields.items():
    value = info.get(field)

    if value is not None:
        valuation[label] = value

valuation_df = pd.DataFrame(
    list(valuation.items()),
    columns=["Metric", "Value"]
)

print("Yahoo Finance valuation snapshot")
display(valuation_df)


In [ ]:
import time
import random
from google.genai import errors

def generate_with_retry(prompt, model="gemini-3.8-flash", max_retries=5):
    """
    Retry Gemini requests when the server is temporarily unavailable.
    """
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=model,
                contents=prompt
            )
            return response.text

        except Exception as e:
            error_message = str(e)
            is_temporary = (
                "503" in error_message
                or "UNAVAILABLE" in error_message
                or "429" in error_message
                or "RESOURCE_EXHAUSTED" in error_message
                or "500" in error_message
                or "502" in error_message
                or "504" in error_message
            )

            if not is_temporary:
                raise

            if attempt == max_retries - 1:
                print("Gemini is still unavailable after several attempts.")
                print("Please wait a few minutes and run the report again.")
                raise

            # Increasing wait time: approximately 2, 4, 8, 16 seconds
            wait_seconds = (2 ** (attempt + 1)) + random.uniform(0, 1)
            print(
                f"Temporary Gemini error. "
                f"Retrying in {wait_seconds:.1f} seconds "
                f"(attempt {attempt + 1}/{max_retries})..."
            )
            time.sleep(wait_seconds)


In [ ]:

import time
import random

def generate_fundamental_report():
    question = (
        "Prepare a comprehensive fundamental analysis of "
        "Reliance Industries Limited using the available financial data."
    )

    # Retrieve relevant financial information from FAISS
    retrieved = search_financial_data(question, top_k=10)

    evidence = "\n\n".join(
        [
            f"[Evidence {i+1}] Source: {item['source']}\n"
            f"{item['text']}"
            for i, item in enumerate(retrieved)
        ]
    )

    # Prepare the financial tables for Gemini
    metrics_text = fundamentals.tail(5).to_string()
    valuation_text = valuation_df.to_string(index=False)

    prompt = f"""
You are a financial research analyst.

Analyze Reliance Industries Limited using only the information below.

HISTORICAL FINANCIAL METRICS:
{metrics_text}

VALUATION SNAPSHOT:
{valuation_text}

RETRIEVED FINANCIAL EVIDENCE:
{evidence}

Write the report with these sections:
1. Executive summary
2. Revenue and growth
3. Profitability and earnings
4. Assets, liabilities and equity
5. Debt, cash and net debt
6. Operating cash flow and free cash flow
7. Financial ratios and valuation
8. Historical trends
9. Strengths and risks
10. Overall fundamental assessment

Important:
- Do not invent missing values or facts.
- Mention when data is unavailable.
- Distinguish historical financial results from current valuation metrics.
- Explain the reporting periods where possible.
- This is informational analysis, not personalized investment advice.
"""

    # Retry temporary Gemini server errors
    for attempt in range(5):
        try:
            response = client.models.generate_content(
                model="gemini-3.8-flash",
                contents=prompt
            )
            return response.text

        except Exception as e:
            error_message = str(e)

            temporary_error = any(
                code in error_message
                for code in [
                    "503", "UNAVAILABLE",
                    "429", "RESOURCE_EXHAUSTED",
                    "500", "502", "504"
                ]
            )

            if not temporary_error or attempt == 4:
                print("Gemini request failed:", error_message)
                raise

            wait_seconds = (2 ** (attempt + 1)) + random.uniform(0, 1)
            print(
                f"Temporary error. Retrying in "
                f"{wait_seconds:.1f} seconds..."
            )
            time.sleep(wait_seconds)


# Call the function OUTSIDE its definition
report = generate_fundamental_report()
print(report)


In [ ]:

import pandas as pd
import time
import random

def generate_fundamental_report():
    # 1. Prepare historical metrics
    metrics_text = fundamentals.tail(5).to_string()

    # 2. Prepare valuation metrics
    valuation_text = valuation_df.to_string(index=False)

    # 3. Retrieve relevant evidence from FAISS
    question = """
    Analyze Reliance Industries Limited fundamentals:
    revenue growth, net income, EPS, assets, liabilities,
    debt, cash, operating cash flow, free cash flow,
    financial ratios, historical performance, strengths and risks.
    """

    retrieved = search_financial_data(question, top_k=10)

    evidence = "\n\n".join(
        f"[Evidence {i+1}] Source: {item['source']}\n{item['text']}"
        for i, item in enumerate(retrieved)
    )

    # 4. Build the analysis prompt
    prompt = f"""
You are a financial research analyst analyzing Reliance Industries
Limited (NSE ticker: RELIANCE.NS).

Use the supplied financial metrics and retrieved evidence to prepare
a comprehensive fundamental analysis report.

HISTORICAL FINANCIAL METRICS:
{metrics_text}

VALUATION METRICS:
{valuation_text}

RETRIEVED EVIDENCE:
{evidence}

Cover ALL the following sections:

1. Revenue and Revenue Growth
   - Revenue across available periods
   - Growth rates and possible trends

2. Profitability and Net Income
   - Net income and operating income
   - Net profit margin and operating margin

3. Earnings Per Share (EPS)
   - EPS across available periods
   - Growth or decline, if data is available

4. Assets and Liabilities
   - Total assets, liabilities and shareholders' equity
   - Changes in the balance sheet

5. Debt and Cash Position
   - Total debt, cash and net debt
   - Debt-to-equity ratio and financial risk

6. Operating Cash Flow and Free Cash Flow
   - Operating cash flow
   - Capital expenditure and free cash flow
   - Whether cash generation supports the business

7. Financial Ratios
   - Profitability, liquidity, leverage and valuation ratios
   - Explain what each available ratio indicates

8. Historical Performance
   - Compare the available reporting periods
   - Identify positive and negative trends

9. Key Financial Strengths and Risks
   - List strengths supported by the data
   - List risks and areas requiring further investigation

10. Overall Fundamental Assessment
    - Summarize the company's financial position
    - Distinguish observed facts from interpretation

IMPORTANT RULES:
- Use only the supplied figures and evidence.
- Do not invent values or missing information.
- State the reporting period and units when available.
- Do not treat missing data as zero.
- Clearly flag unavailable or inconsistent metrics.
- Explain that valuation ratios are a snapshot, not historical
  values unless historical data is supplied.
- Cite retrieved passages using [Evidence 1], [Evidence 2], etc.
- Do not claim that a source supports a statement unless it does.
- This is educational analysis, not personalized investment advice.
"""

    # 5.Generate the report with longer retries
    for attempt in range(7):
        try:
            response = client.models.generate_content(
                model="gemini-3.8-flash",
                contents=prompt
            )

            if response.text:
                return response.text

            raise ValueError("Gemini returned an empty response.")

        except Exception as e:
            error_text = str(e)

            temporary_error = any(
                code in error_text
                for code in [
                    "503", "UNAVAILABLE",
                    "429", "RESOURCE_EXHAUSTED",
                    "500", "502", "504"
                ]
            )

            if not temporary_error:
                raise

            if attempt == 6:
                print("Gemini is still unavailable.")
                print("Wait a few minutes and run the report again.")
                raise

            wait_seconds = min(5 * (2 ** attempt), 60) + random.uniform(0, 2)

            print(
                f"Gemini is temporarily busy. "
                f"Retrying in {wait_seconds:.1f} seconds "
                f"(attempt {attempt + 1}/7)..."
            )

            time.sleep(wait_seconds)



# 6. Generate and display the report
report = generate_fundamental_report()
print(report)


In [ ]:
with open("reliance_fundamental_analysis.txt", "w", encoding="utf-8") as file:
    file.write(report)

print("Report saved successfully.")

In [ ]:
import pandas as pd
import time
import random

test_questions = [
    {
        "category": "Revenue",
        "question": "How has Reliance Industries' revenue changed across the available periods?"
    },
    {
        "category": "Profitability",
        "question": "What do the available financial metrics show about net income and profit margins?"
    },
    {
        "category": "EPS",
        "question": "What EPS values are available, and what trend can be established?"
    },
    {
        "category": "Balance Sheet",
        "question": "What do the available data show about total assets, liabilities, and equity?"
    },
    {
        "category": "Debt and Cash",
        "question": "What are the available figures for debt, cash, and net debt?"
    },
    {
        "category": "Cash Flow",
        "question": "What do the available data show about operating cash flow and free cash flow?"
    },
    {
        "category": "Financial Ratios",
        "question": "Which financial ratios are available, and what do they indicate?"
    },
    {
        "category": "Risks",
        "question": "What financial strengths and risks are supported by the retrieved evidence?"
    },
    {
        "category": "Unanswerable Test",
        "question": "What was Reliance Industries' exact daily closing share price on 1 January 1995?"
    }
]

print("Test questions prepared:", len(test_questions))
pd.DataFrame(test_questions)


In [ ]:
import time
import random

def call_gemini_safely(prompt, model="gemini-3.8-flash", retries=5):
    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model=model,
                contents=prompt
            )
            if response.text:
                return response.text
            raise ValueError("Gemini returned an empty response.")

        except Exception as e:
            message = str(e)
            temporary = any(
                code in message
                for code in ["503", "UNAVAILABLE", "429",
                             "RESOURCE_EXHAUSTED", "500", "502", "504"]
            )

            if not temporary or attempt == retries - 1:
                print("Request failed:", message)
                return None

            delay = min(5 * (2 ** attempt), 60) + random.uniform(0, 2)
            print(f"Temporary error. Waiting {delay:.1f} seconds...")
            time.sleep(delay)

answer = call_gemini_safely(
    "Reply with one sentence confirming that the financial analysis assistant is available."
)
print(answer if answer else "Gemini is still unavailable. Try again later.")


In [ ]:
def ask_grounded_question(question, top_k=5):
    # Step 1: Retrieve relevant passages
    results = search_financial_data(question, top_k=top_k)

    # Step 2: Handle missing retrieval results
    if not results:
        return {
            "question": question,
            "retrieved": [],
            "answer": (
                "I cannot answer this from the available "
                "financial knowledge base."
            )
        }

    # Step 3: Prepare retrieved evidence
    evidence = "\n\n".join(
        f"[Evidence {i}]\n"
        f"Source: {item.get('source', 'Unknown')}\n"
        f"Content: {item.get('text', '')}"
        for i, item in enumerate(results, start=1)
    )

    # Step 4: Define prompt BEFORE calling Gemini
    prompt = f"""
You are a financial research assistant.

Question:
{question}

Retrieved financial evidence:
{evidence}

Instructions:
- Answer only from the evidence provided.
- Do not invent financial figures or dates.
- If the evidence is insufficient, say so clearly.
- Cite supporting passages using [Evidence 1], [Evidence 2], etc.
- Distinguish facts from interpretations.
"""

    # Step 5: Generate the answer safely
    answer = call_gemini_safely(
        prompt=prompt,
        model="gemini-3.8-flash",
        retries=5
    )

    if answer is None:
        answer = (
            "Gemini is temporarily unavailable. "
            "Please retry this question later."
        )

    return {
        "question": question,
        "retrieved": results,
        "answer": answer
    }


# Step 6: Test one question
result = ask_grounded_question(
    "What financial information is available about Reliance's revenue?"
)

print("QUESTION:")
print(result["question"])

print("\nANSWER:")
print(result["answer"])

print("\nRETRIEVED SOURCES:")
for item in result["retrieved"]:
    print("-", item.get("source", "Unknown"))


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

ticker_symbol = "RELIANCE.NS"
stock_fundamentals = yf.Ticker(ticker_symbol)

income_stmt_f = stock_fundamentals.income_stmt
balance_sheet_f = stock_fundamentals.balance_sheet
cash_flow_f = stock_fundamentals.cashflow

print("Income statement shape:", income_stmt_f.shape)
print("Balance sheet shape:", balance_sheet_f.shape)
print("Cash flow shape:", cash_flow_f.shape)

display(income_stmt_f)
display(balance_sheet_f)
display(cash_flow_f)


In [ ]:
import pandas as pd
import numpy as np

def find_metric(statement, possible_names):
    if statement is None or statement.empty:
        return None

    # Check exact matches first
    for name in possible_names:
        if name in statement.index:
            return statement.loc[name]

    # Check partial matches
    for row_name in statement.index:
        for name in possible_names:
            if name.lower() in str(row_name).lower():
                return statement.loc[row_name]

    return None

print("find_metric() function created successfully!")


In [ ]:

# Initialize the fundamental documents list
fundamental_documents = []

# Initialize the ratio documents list
ratio_documents = []

print("fundamental_documents initialized successfully!")


In [ ]:
import yfinance as yf

ticker_symbol = "RELIANCE.NS"
stock_fundamentals = yf.Ticker(ticker_symbol)

income_stmt_f = stock_fundamentals.income_stmt
balance_sheet_f = stock_fundamentals.balance_sheet
cash_flow_f = stock_fundamentals.cashflow

print("Financial statements loaded successfully!")

In [ ]:

def find_metric(statement, possible_names):
    if statement is None or statement.empty:
        return None

    # Exact match first
    for name in possible_names:
        if name in statement.index:
            return statement.loc[name]

    # Case-insensitive partial match
    for row_name in statement.index:
        for name in possible_names:
            if name.lower() in str(row_name).lower():
                return statement.loc[row_name]

    return None


def format_metric_documents(statement, statement_name, metrics):
    documents = []

    if statement is None or statement.empty:
        return documents

    for metric_name, possible_names in metrics.items():
        values = find_metric(statement, possible_names)

        if values is None:
            documents.append(
                f"{ticker_symbol}: {metric_name} was not available "
                f"in the retrieved {statement_name}."
            )
            continue

        for period, value in values.items():
            if pd.isna(value):
                continue

            date_label = pd.Timestamp(period).strftime("%Y-%m-%d")

            documents.append(
                f"Company: Reliance Industries Limited. "
                f"Ticker: {ticker_symbol}. "
                f"Statement: {statement_name}. "
                f"Financial period ending: {date_label}. "
                f"Metric: {metric_name}. "
                f"Reported value: {float(value):,.2f}. "
                f"Source: Yahoo Finance via yfinance. "
                f"Verify units and reporting basis in the official report."
            )

    return documents


income_metrics = {
    "Revenue": ["Total Revenue", "Operating Revenue"],
    "Net Income": ["Net Income"],
    "Operating Income": ["Operating Income"],
    "Basic EPS": ["Basic EPS"],
    "Diluted EPS": ["Diluted EPS"],
    "EBITDA": ["EBITDA"],
    "Interest Expense": ["Interest Expense"]
}

balance_metrics = {
    "Total Assets": ["Total Assets"],
    "Total Liabilities": [
        "Total Liabilities Net Minority Interest",
        "Total Liabilities"
    ],
    "Shareholders Equity": [
        "Stockholders Equity",
        "Total Stockholder Equity"
    ],
    "Total Debt": ["Total Debt"],
    "Cash and Short-Term Investments": [
        "Cash Cash Equivalents And Short Term Investments",
        "Cash And Cash Equivalents",
        "Cash And Short Term Investments"
    ],
    "Current Assets": ["Current Assets"],
    "Current Liabilities": ["Current Liabilities"]
}

cash_metrics = {
    "Operating Cash Flow": [
        "Operating Cash Flow",
        "Cash Flow From Continuing Operating Activities"
    ],
    "Capital Expenditure": [
        "Capital Expenditure",
        "Capital Expenditures"
    ],
    "Free Cash Flow": ["Free Cash Flow"]
}

fundamental_documents = []

fundamental_documents += format_metric_documents(
    income_stmt_f, "Income Statement", income_metrics
)

fundamental_documents += format_metric_documents(
    balance_sheet_f, "Balance Sheet", balance_metrics
)

fundamental_documents += format_metric_documents(
    cash_flow_f, "Cash Flow Statement", cash_metrics
)

print("Fundamental documents created:", len(fundamental_documents))
print("\nSample document:")
print(fundamental_documents[0] if fundamental_documents else "No data found")


In [ ]:

info_f = stock_fundamentals.info

ratio_documents = []

def add_ratio_document(name, value, explanation):
    if value is not None and pd.notna(value):
        ratio_documents.append(
            f"Company: Reliance Industries Limited. "
            f"Metric: {name}. Value: {float(value):.4f}. "
            f"Interpretation: {explanation} "
            f"Source: Yahoo Finance. Verify reporting date and units."
        )


add_ratio_document(
    "Trailing P/E",
    info_f.get("trailingPE"),
    "Market price relative to trailing earnings per share."
)

add_ratio_document(
    "Forward P/E",
    info_f.get("forwardPE"),
    "Market price relative to estimated future earnings per share."
)

add_ratio_document(
    "Price-to-Book",
    info_f.get("priceToBook"),
    "Market valuation relative to book value."
)

add_ratio_document(
    "Current Ratio",
    info_f.get("currentRatio"),
    "Current assets relative to current liabilities."
)

roe = info_f.get("returnOnEquity")
add_ratio_document(
    "Return on Equity (%)",
    roe * 100 if roe is not None else None,
    "Profitability relative to shareholder equity."
)

roa = info_f.get("returnOnAssets")
add_ratio_document(
    "Return on Assets (%)",
    roa * 100 if roa is not None else None,
    "Profitability relative to total assets."
)

# Calculate historical revenue growth and net profit margin
revenue_series = find_metric(
    income_stmt_f, ["Total Revenue", "Operating Revenue"]
)

net_income_series = find_metric(
    income_stmt_f, ["Net Income"]
)

if revenue_series is not None:
    revenue_series = pd.to_numeric(
        revenue_series, errors="coerce"
    ).dropna().sort_index()

    for i in range(1, len(revenue_series)):
        previous = revenue_series.iloc[i - 1]
        current = revenue_series.iloc[i]

        if previous != 0:
            growth = (current / previous - 1) * 100
            period = revenue_series.index[i]

            ratio_documents.append(
                f"Company: Reliance Industries Limited. "
                f"Metric: Annual revenue growth. "
                f"Period ending: {pd.Timestamp(period).date()}. "
                f"Value: {growth:.2f}%. "
                f"Calculated from consecutive reported annual "
                f"revenue observations."
            )

if revenue_series is not None and net_income_series is not None:
    net_income_series = pd.to_numeric(
        net_income_series, errors="coerce"
    ).dropna()

    for period in revenue_series.index:
        if period in net_income_series.index:
            revenue_value = revenue_series.loc[period]
            income_value = net_income_series.loc[period]

            if revenue_value != 0:
                margin = income_value / revenue_value * 100

                ratio_documents.append(
                    f"Company: Reliance Industries Limited. "
                    f"Metric: Net profit margin. "
                    f"Period ending: {pd.Timestamp(period).date()}. "
                    f"Value: {margin:.2f}%. "
                    f"Calculated as net income divided by revenue."
                )

fundamental_documents.extend(ratio_documents)

print("Total fundamental documents:", len(fundamental_documents))


In [ ]:
!pip install -q faiss-cpu

In [ ]:
import faiss
import numpy as np

print("FAISS installed successfully!")
print("FAISS version:", faiss.__version__)

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the same embedding model used for your FAISS project
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")


In [ ]:

import faiss
import numpy as np

# Check that financial documents exist
if not fundamental_documents:
    raise ValueError("No fundamental documents found. Run Cell 2 and Cell 3.")

# Use the embedding model from your existing project
fundamental_embeddings = embedding_model.encode(
    fundamental_documents,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

# Create the FAISS index
fundamental_index = faiss.IndexFlatIP(
    fundamental_embeddings.shape[1]
)

fundamental_index.add(fundamental_embeddings)

print("FAISS fundamental index created successfully!")
print("Number of indexed documents:", fundamental_index.ntotal)


In [ ]:

def search_fundamental_data(question, top_k=5):
    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = fundamental_index.search(
        query_embedding,
        min(top_k, fundamental_index.ntotal)
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx >= 0:
            results.append({
                "score": float(score),
                "text": fundamental_documents[idx]
            })

    return results


# Test the retrieval system
question = (
    "What are Reliance Industries' revenue growth, "
    "profitability, debt and cash flow trends?"
)

results = search_fundamental_data(question, top_k=5)

for result in results:
    print("Similarity score:", round(result["score"], 3))
    print("Retrieved information:", result["text"])
    print("-" * 60)


In [ ]:

def search_combined_financial_data(question, top_k=5):
    evidence = []

    # Search your existing financial knowledge base
    try:
        existing_results = search_financial_data(
            question, top_k=top_k
        )

        if isinstance(existing_results, str):
            evidence.append(
                "EXISTING FINANCIAL KNOWLEDGE:\n"
                + existing_results
            )

        elif isinstance(existing_results, list):
            old_texts = []

            for item in existing_results:
                if isinstance(item, dict):
                    text_value = (
                        item.get("text")
                        or item.get("content")
                        or item.get("page_content")
                    )

                    if text_value:
                        old_texts.append(str(text_value))
                else:
                    old_texts.append(str(item))

            if old_texts:
                evidence.append(
                    "EXISTING FINANCIAL KNOWLEDGE:\n"
                    + "\n".join(old_texts)
                )

    except Exception as exc:
        print("Existing search warning:", exc)

    # Search the new fundamental-analysis index
    fundamental_results = search_fundamental_data(
        question, top_k=top_k
    )

    if fundamental_results:
        fundamental_context = "\n".join(
            result["text"] for result in fundamental_results
        )

        evidence.append(
            "FUNDAMENTAL ANALYSIS KNOWLEDGE:\n"
            + fundamental_context
        )

    return "\n\n".join(evidence)


print("Combined financial retrieval function created!")


In [ ]:

!pip install -q google-genai


In [ ]:

from google import genai
import os
from getpass import getpass

# Enter your Gemini API key securely
api_key = getpass("Enter your Gemini API key: ")

# Initialize the Gemini client
clt = genai.Client(api_key=api_key)

# Select the model
GEMINI_MODEL = "gemini-2.5-flash"

print("Gemini client initialized successfully!")

In [ ]:

from google import genai
from getpass import getpass
import time

api_key = getpass("Enter your Gemini API key: ")

start = time.time()

clt = genai.Client(api_key=api_key)

print("Gemini client initialized successfully!")
print("Time taken:", round(time.time() - start, 2), "seconds")


In [ ]:

import time

if "client" in globals():
    gemini_client = client
elif "clt" in globals():
    gemini_client = clt
else:
    raise NameError(
        "Gemini client not found. Run your Gemini setup cell first."
    )

GEMINI_MODEL = globals().get(
    "GEMINI_MODEL", "gemini-2.5-flash"
)


def ask_financial_question(question, top_k=5, retries=3):
    # Retrieve relevant financial information
    context = search_combined_financial_data(
        question, top_k=top_k
    )

    if not context.strip():
        return "Insufficient retrieved evidence to answer reliably."

    # Prepare the prompt
    prompt = f"""
You are a financial analysis assistant.

Answer the question using the retrieved evidence below.

QUESTION:
{question}

RETRIEVED FINANCIAL EVIDENCE:
{context}

INSTRUCTIONS:
1. Explain financial metrics in simple language.
2. Include values and financial periods when available.
3. Distinguish reported values from calculated metrics.
4. Do not invent figures or fill missing values with zero.
5. Mention when the retrieved evidence is insufficient.
6. Discuss relevant financial strengths and risks.
7. Do not make unsupported buy or sell recommendations.
8. State when units or reporting periods require verification.

Provide a structured answer with headings.
"""

    # Call Gemini, retrying temporary service errors
    for attempt in range(retries):
        try:
            response = gemini_client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt
            )

            if response.text:
                return response.text

            return "Gemini returned an empty response."

        except Exception as exc:
            error_message = str(exc)

            temporary_error = any(
                code in error_message
                for code in [
                    "503", "UNAVAILABLE",
                    "429", "RESOURCE_EXHAUSTED"
                ]
            )

            if temporary_error and attempt < retries - 1:
                wait_seconds = 2 ** (attempt + 1)
                print(
                    f"Temporary API error. Retrying in "
                    f"{wait_seconds} seconds..."
                )
                time.sleep(wait_seconds)
            else:
                return f"Gemini API error: {error_message}"


print("Gemini financial Q&A function created!")


In [ ]:

question = (
    "Analyze Reliance Industries' revenue growth, net income, "
    "EPS, debt position, operating cash flow and financial risks."
)

answer = ask_financial_question(question)

print("QUESTION:")
print(question)

print("\nGEMINI'S ANSWER:")
print(answer)


In [ ]:

test_questions = [
    "How has revenue changed over the available financial years?",
    "What do net income and profit margins indicate?",
    "What EPS data is available?",
    "What are the company's total assets and liabilities?",
    "How does the company's debt compare with its cash position?",
    "What do operating cash flow and free cash flow indicate?",
    "What do the available financial ratios tell us?",
    "Summarize the company's historical financial performance."
]

for question in test_questions:
    print("\n" + "=" * 70)
    print("QUESTION:", question)
    print("-" * 70)
    print(ask_financial_question(question))


In [ ]:
print("search_financial_data exists:", "search_financial_data" in globals())
print("search_fundamental_data exists:", "search_fundamental_data" in globals())
print("search_combined_financial_data exists:", "search_combined_financial_data" in globals())

In [ ]:
!pip install -q -U google-genai

from google import genai
from getpass import getpass

api_key = getpass("Enter your Gemini API key: ").strip()
clt = genai.Client(api_key=api_key)

print("Gemini client initialized.")

In [ ]:
import time

def test_gemini_with_retry(max_retries=4):
    for attempt in range(max_retries):
        try:
            response = clt.models.generate_content(
                model="gemini-3.8-flash",
                contents="Reply with exactly: Gemini connection successful"
            )
            print(response.text)
            return

        except Exception as e:
            error = str(e)

            if "503" in error or "429" in error or "UNAVAILABLE" in error:
                wait_seconds = 2 ** (attempt + 1)
                print(
                    f"Temporary server issue. "
                    f"Attempt {attempt + 1}/{max_retries}. "
                    f"Waiting {wait_seconds} seconds..."
                )
                time.sleep(wait_seconds)
            else:
                print("Gemini error:", error)
                return

    print("Gemini is still unavailable. Please try again later.")

test_gemini_with_retry()

In [ ]:
results = search_fundamental_data(
    "What do the available financial ratios tell us?",
    top_k=5
)

for item in results:
    print(item["text"])
    print("-" * 50)

In [ ]:

!pip install -q -U google-genai

In [ ]:
!pip install -q google-auth==2.49.0

In [ ]:
%pip install -q "google-genai>=1.0.0,<2.0.0" "google-auth==2.49.0"


In [ ]:

%pip install -q "google-genai>=1.0.0,<2.0.0" "google-auth==2.49.0"


In [ ]:
import google.auth
import google.genai
from google import genai

print("Google Auth:", google.auth.__version__)
print("Gemini library imported successfully.")

In [ ]:
from getpass import getpass
from google import genai

api_key = getpass("Enter your Gemini API key: ")
clt = genai.Client(api_key=api_key)

print("Gemini client initialized successfully.")

In [ ]:

GEMINI_MODEL = "gemini-2.5-flash"

try:
    response = clt.models.generate_content(
        model=GEMINI_MODEL,
        contents="Explain the P/E ratio in stock market analysis in 3 sentences."
    )

    print("Gemini connection successful!")
    print(response.text)

except Exception as e:
    print("Error:", e)


In [ ]:

import time

GEMINI_MODEL = "gemini-3.8-flash"

try:
    start = time.time()

    response = clt.models.generate_content(
        model=GEMINI_MODEL,
        contents="Reply with only: Gemini is working.",
        config={"http_options": {"timeout": 20000}}
    )

    print("Response:", response.text)
    print("Time taken:", round(time.time() - start, 2), "seconds")

except Exception as e:
    print("Request failed:")
    print(type(e).__name__, str(e))


In [ ]:

import time

GEMINI_MODEL = "gemini-3.8-flash"
max_attempts = 3

for attempt in range(max_attempts):
    try:
        print(f"Attempt {attempt + 1} of {max_attempts}...")

        response = clt.models.generate_content(
            model=GEMINI_MODEL,
            contents="Reply with only: Gemini is working."
        )

        print("\nSUCCESS!")
        print(response.text)
        break

    except Exception as e:
        error = str(e)
        print("Error:", error)

        if "503" in error or "UNAVAILABLE" in error:
            if attempt < max_attempts - 1:
                wait_seconds = 10 * (2 ** attempt)
                print(f"Retrying in {wait_seconds} seconds...")
                time.sleep(wait_seconds)
            else:
                print(
                    "Gemini is still unavailable. "
                    "Try again later."
                )
        else:
            print("This is a different error; check the message above.")
            break


In [ ]:

try:
    print("Checking available Gemini models...\n")

    for model in clt.models.list():
        name = getattr(model, "name", None)
        methods = getattr(model, "supported_actions", None)

        if name and "generateContent" in (methods or []):
            print(name)

except Exception as e:
    print("Unable to list models:")
    print(type(e).__name__, str(e))


In [ ]:

GEMINI_MODEL = "gemini-flash-lite-latest"

try:
    print("Testing model:", GEMINI_MODEL)

    response = clt.models.generate_content(
        model=GEMINI_MODEL,
        contents="Reply with only: Gemini is working.",
        config={"http_options": {"timeout": 20000}}
    )

    print("\nSUCCESS!")
    print(response.text)

except Exception as e:
    print("\nRequest failed:")
    print(type(e).__name__, str(e))


In [ ]:

variables_to_check = [
    "index",
    "faiss_index",
    "documents",
    "chunks",
    "embeddings",
    "search_fundamentals",
    "search_financial_data",
    "search_financial_documents"
]

for name in variables_to_check:
    status = "Available" if name in globals() else "Not defined"
    print(f"{name}: {status}")


In [ ]:

%pip install -q yfinance faiss-cpu sentence-transformers


In [ ]:

import yfinance as yf
import pandas as pd
import numpy as np

ticker = yf.Ticker("RELIANCE.NS")

# Company fundamentals
company_info = ticker.info

# Financial statements
income_statement = ticker.financials
balance_sheet = ticker.balance_sheet
cash_flow = ticker.cashflow

print("Company:", company_info.get("longName", "Reliance Industries"))
print("Sector:", company_info.get("sector", "Not available"))
print("Market Cap:", company_info.get("marketCap", "Not available"))
print("P/E Ratio:", company_info.get("trailingPE", "Not available"))
print("Debt-to-Equity:", company_info.get("debtToEquity", "Not available"))
print("\nIncome statement shape:", income_statement.shape)
print("Balance sheet shape:", balance_sheet.shape)
print("Cash flow shape:", cash_flow.shape)


In [ ]:

documents = []

# Create documents from company fundamentals
key_metrics = [
    "currentPrice",
    "marketCap",
    "trailingPE",
    "forwardPE",
    "priceToBook",
    "returnOnEquity",
    "returnOnAssets",
    "profitMargins",
    "operatingMargins",
    "revenueGrowth",
    "earningsGrowth",
    "totalRevenue",
    "netIncomeToCommon",
    "totalDebt",
    "totalCash",
    "debtToEquity",
    "currentRatio",
    "dividendYield",
    "freeCashflow"
]

for metric in key_metrics:
    value = company_info.get(metric)

    if value is not None:
        documents.append(
            f"Company: Reliance Industries Limited. "
            f"Metric: {metric}. Value: {value}."
        )

# Convert financial statement rows into documents
def statement_to_documents(statement, statement_name):
    result = []

    if statement is None or statement.empty:
        return result

    for item in statement.index:
        values = statement.loc[item].dropna()

        if not values.empty:
            details = "; ".join(
                f"{date}: {value}"
                for date, value in values.items()
            )

            result.append(
                f"Company: Reliance Industries Limited. "
                f"Statement: {statement_name}. "
                f"Financial item: {item}. Values: {details}."
            )

    return result

documents.extend(
    statement_to_documents(income_statement, "Income Statement")
)
documents.extend(
    statement_to_documents(balance_sheet, "Balance Sheet")
)
documents.extend(
    statement_to_documents(cash_flow, "Cash Flow Statement")
)

print("Total financial documents:", len(documents))
print("\nSample document:")
print(documents[0] if documents else "No financial documents retrieved.")


In [ ]:

import faiss
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print("FAISS index created successfully!")
print("Number of indexed documents:", index.ntotal)
print("Embedding dimensions:", embeddings.shape[1])


In [ ]:

def search_financial_data(question, top_k=5):
    if index.ntotal == 0:
        return []

    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(
        query_embedding,
        min(top_k, index.ntotal)
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx >= 0:
            results.append({
                "score": float(score),
                "document": documents[idx]
            })

    return results


# Test retrieval
question = "What financial information is available about Reliance's debt?"

results = search_financial_data(question)

for i, result in enumerate(results, start=1):
    print(f"\nResult {i} | Similarity: {result['score']:.3f}")
    print(result["document"])


In [ ]:

GEMINI_MODEL = "gemini-flash-lite-latest"

def ask_financial_question(question, top_k=5):
    retrieved = search_financial_data(question, top_k=top_k)

    if not retrieved:
        return "No relevant financial documents were found."

    context = "\n\n".join(
        item["document"] for item in retrieved
    )

    prompt = f"""
You are a financial analysis assistant.

Answer the question using only the financial evidence below.
Do not invent financial values, dates, or ratios.
If the evidence is insufficient, clearly state what is missing.
Distinguish reported facts from your interpretation.
Mention the reporting period whenever the evidence provides it.

Retrieved financial evidence:
{context}

Question:
{question}

Answer:
"""

    response = clt.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt
    )

    return response.text


# Test the complete FAISS + Gemini pipeline
answer = ask_financial_question(
    "What financial information is available about Reliance Industries' profitability?"
)

print(answer)


In [ ]:

test_questions = [
    "What is Reliance Industries' P/E ratio?",
    "What is the company's market capitalization?",
    "What financial information is available about revenue?",
    "What is the company's net income?",
    "What is its debt-to-equity ratio?",
    "What information is available about cash and total debt?",
    "What does the available evidence indicate about profitability?",
    "What information is available about return on equity?",
    "What is the company's free cash flow?",
    "What financial strengths and risks can be identified from the retrieved data?"
]

evaluation_results = []

for i, question in enumerate(test_questions, start=1):
    print("\n" + "=" * 80)
    print(f"QUESTION {i}: {question}")
    print("=" * 80)

    try:
        # Test retrieval separately
        retrieved = search_financial_data(question, top_k=5)

        print("\n--- RETRIEVED EVIDENCE ---")

        for j, item in enumerate(retrieved, start=1):
            print(f"\nEvidence {j} | Similarity: {item['score']:.3f}")
            print(item["document"])

        # Generate a grounded answer
        answer = ask_financial_question(question, top_k=5)

        print("\n--- GEMINI ANSWER ---")
        print(answer)

        evaluation_results.append({
            "Question": question,
            "Retrieved documents": len(retrieved),
            "Top similarity": (
                round(retrieved[0]["score"], 3)
                if retrieved else None
            ),
            "Answer generated": bool(answer),
            "Answer": answer
        })

    except Exception as e:
        print("\nError:", type(e).__name__, str(e))

        evaluation_results.append({
            "Question": question,
            "Retrieved documents": 0,
            "Top similarity": None,
            "Answer generated": False,
            "Answer": str(e)
        })

print("\nTesting completed.")


In [ ]:

test_questions = [
    "What is Reliance Industries' P/E ratio?",
    "What financial information is available about revenue?",
    "What financial information is available about debt and cash?"
]

evaluation_results = []

for question in test_questions:
    print("\n" + "=" * 60)
    print("QUESTION:", question)

    # Retrieve relevant financial evidence
    retrieved = search_financial_data(question, top_k=3)

    print("\nRetrieved documents:", len(retrieved))

    for i, item in enumerate(retrieved, start=1):
        print(f"\nEvidence {i} (score: {item['score']:.3f}):")
        print(item["document"])

    # Generate an answer using the retrieved evidence
    try:
        answer = ask_financial_question(question, top_k=3)
        print("\nGEMINI ANSWER:")
        print(answer)

        evaluation_results.append({
            "Question": question,
            "Retrieved documents": len(retrieved),
            "Answer generated": True,
            "Answer": answer
        })

    except Exception as e:
        print("\nERROR:", type(e).__name__, str(e))

        evaluation_results.append({
            "Question": question,
            "Retrieved documents": len(retrieved),
            "Answer generated": False,
            "Answer": str(e)
        })


In [ ]:

results_df = pd.DataFrame(evaluation_results)

display(
    results_df[
        ["Question", "Retrieved documents", "Answer generated"]
    ]
)


In [ ]:

print("Total questions tested:", len(results_df))

print(
    "Successful answers:",
    results_df["Answer generated"].sum()
)

print(
    "Failed answers:",
    (~results_df["Answer generated"]).sum()
)

display(results_df)


In [ ]:

question = "What is Reliance Industries' P/E ratio?"

# First, inspect the evidence without calling Gemini
retrieved = search_financial_data(question, top_k=3)

print("Retrieved evidence for the P/E ratio:")

for i, item in enumerate(retrieved, start=1):
    print(f"\nEvidence {i}:")
    print(item["document"])

# Retry Gemini once
if retrieved:
    context = "\n\n".join(
        item["document"] for item in retrieved
    )

    prompt = f"""
Answer the question using only the evidence provided.
Do not invent financial figures.
If the P/E ratio is not explicitly available, say so.
Mention that the ratio may change with the share price.

Evidence:
{context}

Question: {question}
"""

    try:
        response = clt.models.generate_content(
            model="gemini-flash-lite-latest",
            contents=prompt,
            config={"http_options": {"timeout": 20000}}
        )

        print("\nGemini answer:")
        print(response.text)

    except Exception as e:
        print("\nRetry failed:", type(e).__name__, str(e))
        print("Keep the existing FAISS index and try again later.")
else:
    print("No evidence found. Check your financial documents.")


In [ ]:

# Step 11: Add valuation metrics to the FAISS knowledge base

# Yahoo Finance valuation metrics
valuation_metrics = {
    "trailingPE": "Trailing P/E ratio (price-to-earnings ratio based on past earnings)",
    "forwardPE": "Forward P/E ratio (price-to-earnings ratio based on estimated future earnings)",
    "priceToBook": "Price-to-book ratio",
    "enterpriseToEbitda": "Enterprise value to EBITDA ratio",
    "enterpriseToRevenue": "Enterprise value to revenue ratio",
    "marketCap": "Market capitalization",
    "currentPrice": "Current share price"
}

# Retrieve the latest available Yahoo Finance information
company_info = ticker.info

new_documents = []

for metric, description in valuation_metrics.items():
    value = company_info.get(metric)

    # Skip missing or invalid values
    if value is not None and pd.notna(value):
        doc = (
            f"Company: Reliance Industries Limited. "
            f"Financial metric: {description}. "
            f"Metric key: {metric}. Value: {value}. "
            f"Search terms: P/E ratio, valuation, financial ratios, "
            f"Reliance Industries, {metric}."
        )
        new_documents.append(doc)

# Avoid adding duplicate documents if you run this cell again
existing_documents = set(documents)
new_documents = [
    doc for doc in new_documents
    if doc not in existing_documents
]

if new_documents:
    new_embeddings = embedding_model.encode(
        new_documents,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    ).astype("float32")

    index.add(new_embeddings)
    documents.extend(new_documents)

    print(f"Added {len(new_documents)} new financial documents.")
else:
    print("No new valuation documents added.")

print("\nAvailable valuation metrics:")
for doc in documents:
    if any(metric in doc for metric in valuation_metrics):
        print("-", doc)


In [ ]:

question = "What is Reliance Industries' P/E ratio?"

results = search_financial_data(question, top_k=5)

print("Retrieved documents:")
for i, item in enumerate(results, start=1):
    print(f"\n{i}. Similarity score: {item['score']:.3f}")
    print(item["document"])

print("\nGemini answer:")
print(ask_financial_question(question, top_k=5))


In [ ]:

# Step 13: Evaluate the financial Q&A system

test_questions = [
    "What is Reliance Industries' trailing and forward P/E ratio?",
    "What financial information is available about revenue?",
    "What financial information is available about debt and cash?",
    "What is Reliance Industries' market capitalization?",
    "What is the price-to-book ratio of Reliance Industries?"
]

evaluation_results = []

for question in test_questions:
    print("\n" + "=" * 70)
    print("QUESTION:", question)

    retrieved = search_financial_data(question, top_k=5)

    print("\nRETRIEVED EVIDENCE:")
    for item in retrieved:
        print("-", item["document"])

    try:
        answer = ask_financial_question(question, top_k=5)
        success = bool(answer and answer.strip())

    except Exception as e:
        answer = f"API error: {type(e).__name__}: {e}"
        success = False

    print("\nANSWER:")
    print(answer)

    evaluation_results.append({
        "Question": question,
        "Retrieved documents": len(retrieved),
        "Answer generated": success,
        "Answer": answer
    })


In [ ]:

# Step 14: Display the evaluation results

results_df = pd.DataFrame(evaluation_results)

display(
    results_df[
        ["Question", "Retrieved documents", "Answer generated"]
    ]
)

successful = results_df["Answer generated"].sum()
total = len(results_df)

print(f"Total questions tested: {total}")
print(f"Successful answers: {successful}")
print(f"Failed answers: {total - successful}")
print(f"Generation success rate: {successful / total:.1%}")
